In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "24"
os.environ["OPENBLAS_NUM_THREADS"] = "24"
os.environ["VECLIB_MAXIMUM_THREADS"] = "24"
os.environ["NUMEXPR_NUM_THREADS"] = "24"



import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("numpy:",np.__version__)
print("torch:", torch.__version__)

print("logical CPUs:", os.cpu_count())
print("torch num threads:", torch.get_num_threads())
print("torch interop threads:", torch.get_num_interop_threads())


numpy: 1.26.4
torch: 2.2.2
logical CPUs: 24
torch num threads: 24
torch interop threads: 2


In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "24"
os.environ["OPENBLAS_NUM_THREADS"] = "24"
os.environ["VECLIB_MAXIMUM_THREADS"] = "24"
os.environ["NUMEXPR_NUM_THREADS"] = "24"



import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("numpy:",np.__version__)
print("torch:", torch.__version__)

print("logical CPUs:", os.cpu_count())
print("torch num threads:", torch.get_num_threads())
print("torch interop threads:", torch.get_num_interop_threads())


numpy: 1.24.4
torch: 2.3.0.dev20240109
logical CPUs: 8
torch num threads: 24
torch interop threads: 8


In [ ]:
TRAIN_X_PATH = "../Data/outputs_train.npy"
TRAIN_Y_PATH = "../Data/ks_train.npy"
VAL_X_PATH   = "../Data/outputs_val.npy"
VAL_Y_PATH   = "../Data/ks_val.npy"
TEST_X_PATH  = "../Data/outputs_test.npy"
TEST_Y_PATH  = "../Data/ks_test.npy"

SEED = 42
BATCH_SIZE = 32
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-6
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
PATIENCE = 20

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

device = cpu


## Load data

In [ ]:
x_train = np.load(TRAIN_X_PATH).astype(np.float32)   # (800, 11993)
y_train = np.load(TRAIN_Y_PATH).astype(np.float32)   # (800, 6)

x_val = np.load(VAL_X_PATH).astype(np.float32)       # (100, 11993)
y_val = np.load(VAL_Y_PATH).astype(np.float32)       # (100, 6)

x_test = np.load(TEST_X_PATH).astype(np.float32)     # (100, 11993)
y_test = np.load(TEST_Y_PATH).astype(np.float32)     # (100, 6)

print("x_train:", x_train.shape, "y_train:", y_train.shape)
print("x_val  :", x_val.shape,   "y_val  :", y_val.shape)
print("x_test :", x_test.shape,  "y_test :", y_test.shape)

x_train: (800, 11993) y_train: (800, 6)
x_val  : (100, 11993) y_val  : (100, 6)
x_test : (100, 11993) y_test : (100, 6)


## Preprocessing

In [ ]:
# Input scaling using training statistics only
x_mean = x_train.mean(dtype=np.float64)
x_std = x_train.std(dtype=np.float64) + 1e-8

x_train_s = (x_train - x_mean) / x_std
x_val_s   = (x_val   - x_mean) / x_std
x_test_s  = (x_test  - x_mean) / x_std

# Target scaling:
# log10 first, then standardize column-wise
y_train_log = np.log10(y_train)
y_val_log   = np.log10(y_val)
y_test_log  = np.log10(y_test)

y_mean = y_train_log.mean(axis=0, keepdims=True)
y_std  = y_train_log.std(axis=0, keepdims=True) + 1e-8

y_train_s = (y_train_log - y_mean) / y_std
y_val_s   = (y_val_log   - y_mean) / y_std
y_test_s  = (y_test_log  - y_mean) / y_std

def inverse_target_transform(y_scaled):
    """
    Convert standardized-log targets back to original ks scale.
    """
    y_log = y_scaled * y_std + y_mean
    return 10.0 ** y_log

## Dataset / DataLoader

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, x, y):
        # x: (N, 11993) -> (N, 11993, 1)
        self.x = torch.from_numpy(x).float().unsqueeze(-1)
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_ds = SequenceDataset(x_train_s, y_train_s)
val_ds   = SequenceDataset(x_val_s, y_val_s)
test_ds  = SequenceDataset(x_test_s, y_test_s)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

## RNN model

In [ ]:
class RNNRegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=6, dropout=0.2):
        super().__init__()

# To switch from RNN to GRU, only change this part:
        """        
        self.rnn = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        """
        self.rnn = nn.RNN(                                  
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            nonlinearity="tanh",
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        _, h_n = self.rnn(x)     # h_n: (num_layers, batch, hidden_size)
        last_hidden = h_n[-1]    # (batch, hidden_size)
        y = self.head(last_hidden)
        return y

model = RNNRegressor(
    input_size=1,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    output_size=6,
    dropout=DROPOUT,
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

## Training / evaluation functions

In [ ]:
def run_one_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_count = 0

    preds_scaled = []
    targets_scaled = []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            pred = model(xb)
            loss = criterion(pred, yb)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        total_count += bs

        preds_scaled.append(pred.detach().cpu().numpy())
        targets_scaled.append(yb.detach().cpu().numpy())

    avg_loss = total_loss / total_count
    preds_scaled = np.vstack(preds_scaled)
    targets_scaled = np.vstack(targets_scaled)

    return avg_loss, preds_scaled, targets_scaled

def compute_metrics(pred_scaled, true_scaled, name="set"):
    pred_orig = inverse_target_transform(pred_scaled)
    true_orig = inverse_target_transform(true_scaled)

    mse = np.mean((pred_orig - true_orig) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(pred_orig - true_orig))

    relative_error = np.abs(pred_orig - true_orig) / (np.abs(true_orig) + 1e-12)
    mre_percent = np.mean(relative_error) * 100.0
    per_param_mre_percent = np.mean(relative_error, axis=0) * 100.0

    print(f"\n[{name}] original-scale metrics")
    print(f"  RMSE = {rmse:.6e}")
    print(f"  MAE  = {mae:.6e}")
    print(f"  MRE% = {mre_percent:.4f}")
    print(f"  Per-parameter MRE% = {per_param_mre_percent}")

    return {
        "rmse": rmse,
        "mae": mae,
        "mre_percent": mre_percent,
        "per_param_mre_percent": per_param_mre_percent,
    }

## Training loop with early stopping

In [ ]:
best_state = None
best_val_loss = float("inf")
best_epoch = -1
wait = 0

history = {
    "train_loss": [],
    "val_loss": [],
}

for epoch in range(1, EPOCHS + 1):
    train_loss, _, _ = run_one_epoch(model, train_loader, optimizer)
    val_loss, _, _   = run_one_epoch(model, val_loader, optimizer=None)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        wait = 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        wait += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train_loss = {train_loss:.6f} | val_loss = {val_loss:.6f}")

    if wait >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        print(f"Best epoch = {best_epoch}, best val loss = {best_val_loss:.6f}")
        break

KeyboardInterrupt: 

## Load best model

In [ ]:
model.load_state_dict(best_state)

## Final evaluation

In [ ]:
train_loss, train_pred_s, train_true_s = run_one_epoch(model, train_loader, optimizer=None)
val_loss, val_pred_s, val_true_s       = run_one_epoch(model, val_loader, optimizer=None)
test_loss, test_pred_s, test_true_s    = run_one_epoch(model, test_loader, optimizer=None)

print("\nBest epoch =", best_epoch)
print(f"Best validation loss (scaled MSE) = {best_val_loss:.6f}")
print(f"Final train loss (scaled MSE) = {train_loss:.6f}")
print(f"Final val loss   (scaled MSE) = {val_loss:.6f}")
print(f"Final test loss  (scaled MSE) = {test_loss:.6f}")

train_metrics = compute_metrics(train_pred_s, train_true_s, name="train")
val_metrics   = compute_metrics(val_pred_s, val_true_s, name="val")
test_metrics  = compute_metrics(test_pred_s, test_true_s, name="test")

## Save predictions and model

In [ ]:
np.save("train_pred_ks.npy", inverse_target_transform(train_pred_s))
np.save("val_pred_ks.npy",   inverse_target_transform(val_pred_s))
np.save("test_pred_ks.npy",  inverse_target_transform(test_pred_s))
torch.save(best_state, "best_rnn_regressor.pt")

print("\nSaved files:")
print("  best_rnn_regressor.pt")
print("  train_pred_ks.npy")
print("  val_pred_ks.npy")
print("  test_pred_ks.npy")